# Units of memory: converter and self-quiz (Chapter 1)

Use this notebook to check your own answers to unit questions, and to practise.

**The rule (section 1.7):** 4 bits = 1 nibble; 8 bits = 1 byte; KB, MB, GB, TB, PB: each **1024** times the one before. Big → small: **multiply**; small → big: **divide**.

**How to run:** click a grey cell, then press **Shift + Enter**. Run the first cell once.

In [ ]:
# Run this cell once. It sets up the units of Chapter 1, section 1.7.
from fractions import Fraction
import random

UNITS = ['bit', 'nibble', 'byte', 'KB', 'MB', 'GB', 'TB', 'PB']
BITS = {'bit': 1, 'nibble': 4, 'byte': 8}
for k, u in enumerate(['KB', 'MB', 'GB', 'TB', 'PB']):
    BITS[u] = 8 * 1024 ** (k + 1)          # every unit from KB to PB is 1024 times the one before

def unit_of(name):
    for u in UNITS:
        if name.strip().lower().rstrip('s') == u.lower():
            return u
    raise ValueError('Unknown unit: ' + name + '. Use one of: ' + ', '.join(UNITS))

def nice(x):
    x = Fraction(x)
    if x.denominator == 1:
        return f'{x.numerator:,}'.replace(',', ' ')
    return f'{float(x):.6g}'

def convert(amount, frm, to, show=True):
    """Convert an amount from one unit to another, showing the steps of the ladder."""
    frm, to = unit_of(frm), unit_of(to)
    value = Fraction(str(amount)) * BITS[frm] / BITS[to]
    if show:
        ladder = UNITS if 'nibble' in (frm, to) else [u for u in UNITS if u != 'nibble']   # bit -> byte is one step: x 8
        i, j = ladder.index(frm), ladder.index(to)
        step = 1 if j > i else -1
        steps = []
        for k in range(i, j, step):
            a, b = ladder[k], ladder[k + step]
            f = BITS[max(a, b, key=BITS.get)] // BITS[min(a, b, key=BITS.get)]
            steps.append(('÷ ' if step > 0 else '× ') + str(f) + f' ({a} → {b})')
        print(f'{nice(amount)} {frm} = {nice(value)} {to}')
        if steps:
            print('   steps:', ', then '.join(steps))
        print('   rule:', 'big → small: multiply' if j < i else 'small → big: divide' if j > i else 'same unit')
    return value

def arrange(*sizes, descending=False):
    """Arrange sizes given as strings like '2 GB', '2100 MB'."""
    parsed = []
    for s in sizes:
        n, u = s.split()
        parsed.append((Fraction(n) * BITS[unit_of(u)], s))
    parsed.sort(reverse=descending)
    out = parsed[0][1]
    for (b0, _), (b1, s1) in zip(parsed, parsed[1:]):
        out += (' = ' if b0 == b1 else ' > ' if descending else ' < ') + s1
    print(out)
    print('   working (all in bytes):', ', '.join(f'{s} = {nice(b / 8)} bytes' for b, s in parsed))

def how_many_fit(space, file_size):
    """How many files of one size fit in a space? Sizes as strings, e.g. '64 GB', '4 MB'."""
    n1, u1 = space.split(); n2, u2 = file_size.split()
    total = Fraction(n1) * BITS[unit_of(u1)]
    each = Fraction(n2) * BITS[unit_of(u2)]
    k = total // each
    print(f'{nice(k)} files of {file_size} fit in {space}')
    print(f'   working: change both to one unit, then divide and keep the whole number: {nice(total / 8)} bytes ÷ {nice(each / 8)} bytes')
    return k

print('Ready. Units:', ' < '.join(UNITS))

## 1. Convert
Change the numbers and units between the brackets, then run the cell. The steps of the ladder are shown.

In [ ]:
convert(3, 'GB', 'KB')
convert(24576, 'bit', 'KB')
convert(2, 'PB', 'GB')

## 2. Arrange sizes in order
Give each size as a number, a space and a unit.

In [ ]:
arrange('2 GB', '2100 MB', '2000000 KB')
arrange('1 TB', '1000 GB', '1048576 MB', descending=True)

## 3. How many files fit?

In [ ]:
how_many_fit('64 GB', '4 MB')
how_many_fit('1 TB', '700 MB')

## 4. Self-quiz
Five conversion questions; type a whole number and press Enter. You are told at once whether it is right, with the steps.

In [ ]:
# Self-quiz: run this cell; answer each question in the box that appears. Run it again for new questions.
def quiz(n=5, seed=None):
    rnd = random.Random(seed)
    score = 0
    for q in range(1, n + 1):
        kind = rnd.choice(['down', 'up', 'bits'])
        if kind == 'down':                      # big unit -> small unit: a whole number
            i = rnd.randrange(4, 8); j = i - rnd.choice([1, 2])
            amount = rnd.randrange(2, 13)
        elif kind == 'up':                      # small -> big, chosen so the answer is whole
            j = rnd.randrange(4, 8); i = j - rnd.choice([1, 2])
            amount = rnd.randrange(2, 13) * 1024 ** (j - i)
        else:                                   # bytes or KB -> bits
            i, j = rnd.choice([2, 3]), 0
            amount = rnd.randrange(2, 13)
        frm, to = UNITS[i], UNITS[j]
        right = convert(amount, frm, to, show=False)
        reply = input(f'Q{q}. {nice(amount)} {frm} = ? {to}   ').replace(' ', '').replace(',', '')
        try:
            ok = Fraction(reply) == right
        except (ValueError, ZeroDivisionError):
            ok = False
        score += ok
        print('   Correct!' if ok else '   Not quite.', end=' ')
        convert(amount, frm, to)
    print(f'\nScore: {score} out of {n}')

quiz()

## Try these
1. How many bits are in 1 MB?
2. Arrange in ascending order: 1 GB, 1000 MB, 1048576 KB.
3. How many 5 MB photos fit on a 32 GB card?
4. Convert 3 145 728 KB into GB.

*More questions with answers: the Chapter 1 question pages on the book's website.*